# 28 — Prompt Architecture Patterns and System Selection

    ## Scenario and success criteria

    Northstar selects the least complex architecture that meets auditability, adaptability, latency, and operating-cost needs.

    This guided lab succeeds when its assertions pass and the learner can explain why the baseline fails, what the mitigation changes, and which production controls remain outside the simulation.

    ## Learning objectives

    - Turn requirements into explicit criteria.
- Expose the score behind a recommendation.
- Test whether the winner changes under plausible weights.

    **Prerequisites:** Courses 01–13 and the preceding advanced/enterprise lesson.
    **Safety boundary:** all behavior is deterministic and synthetic; there are no credentials, external calls, or side effects. Printed results are simulation evidence, not a live-model benchmark.

## Mental model and architecture

![Course 28 architecture](diagram-1.svg)

Treat the model as one uncertain component inside a deterministic control plane. Inputs, schemas, identity, authorization, metrics, release gates, and state transitions remain application responsibilities.

## Baseline and failure injection

A single weighted score can disguise fragile assumptions and false precision.

The next cell defines the synthetic fixture and the smallest reusable primitive needed to make that failure observable.

In [ ]:
from lab28 import Architecture, select_architecture, sensitivity

options = [
    Architecture("single-call", 4, 1, 5, 3, 5),
    Architecture("workflow", 5, 3, 4, 5, 3),
    Architecture("agent", 2, 5, 2, 2, 1),
]
weights = {"determinism": 3, "adaptability": 1, "latency": 2, "auditability": 3, "operational_cost": 1}

## Experiment

Run the baseline and candidate on the same fixture so the comparison is attributable.

In [ ]:
winner, scores = select_architecture(options, weights)
print("winner", winner)
print("scores", scores)
weight_sets = [weights, {**weights, "adaptability": 6}, {**weights, "operational_cost": 5}]
print("sensitivity", sensitivity(options, weight_sets))

## Evaluation

The assertions below are the executable contract. They validate both a positive path and a boundary or failure path; a printed claim alone is not proof.

In [ ]:
assert winner == "workflow"
assert sum(sensitivity(options, weight_sets).values()) == len(weight_sets)
assert set(scores) == {item.name for item in options}

## Production upgrade

Prototype the top two options on representative cases. Include failure recovery, authorization, observability, skill availability, team capacity, and total operating cost—not only model quality.

| Teaching lab | Production system |
| --- | --- |
| Synthetic fixtures | Versioned, reviewed, privacy-safe datasets |
| Deterministic simulation | Provider adapter plus optional recorded replay |
| In-process state | Durable state with tenant and retention boundaries |
| Assertions | CI gates, staged rollout, monitoring, and rollback |

## Exercises

1. Add one normal, one boundary, and one adversarial case without weakening an invariant.
2. Change one design variable and report the metric numerator, denominator, unit, and direction.
3. Write a production decision memo that identifies owner, failure policy, monitoring signal, and rollback trigger.

## Takeaway

Use probabilistic components for bounded interpretation; use trusted deterministic code for permissions, validation, metrics, and consequential state changes.